In [1]:
import pandas as pd
import numpy as np

In [2]:
pd.set_option('display.max_columns', None) 

In [3]:
applications = pd.read_csv('h3_applications.csv')

In [4]:
industries = pd.read_csv('h3_industries.csv')

In [5]:
applications = applications.drop_duplicates('applicant_id')

In [6]:
applications['External Rating'] = applications['External Rating'].fillna(0)

In [7]:
applications['Education level'] = applications['Education level'].fillna('Середня')

In [8]:
# Exclude applications with missing amount or no external rating
applications = applications[(applications['External Rating'] != 0) & 
                            (applications['Amount'] != 0) &
                            (applications['Amount'].notna())]

In [9]:
applications = applications.merge(industries, how = 'left', on = 'Industry')

In [10]:
applications['Applied at'] = pd.to_datetime(applications['Applied at'], format = 'mixed')

In [11]:
applications['Applied day'] = applications['Applied at'].dt.day_name()

In [12]:
applications.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12722 entries, 0 to 12721
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Applied at       12722 non-null  datetime64[ns]
 1   Amount           12722 non-null  float64       
 2   Age              12722 non-null  int64         
 3   Gender           12722 non-null  object        
 4   Industry         12722 non-null  object        
 5   Marital status   12722 non-null  object        
 6   External Rating  12722 non-null  float64       
 7   Education level  12722 non-null  object        
 8   Location         11034 non-null  object        
 9   applicant_id     12722 non-null  object        
 10  Score            12722 non-null  int64         
 11  Applied day      12722 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(2), object(7)
memory usage: 1.2+ MB


In [13]:
# Score is 0 if External Rating == 0 or Amount is missing
applications['total_score'] = ((applications['Age'].between(35, 55)) * 20 +
                               (~applications['Applied day'].isin(['Saturday', 'Sunday'])) * 20 +
                               (applications['Marital status'] == 'Married') * 20 +
                               (applications['Location'] == 'Київ чи область') * 10 +
                               (applications['Score']) + 
                               (applications['External Rating'] >= 7) * 20 +
                               (applications['External Rating'] <= 2) * (-20)
                              )

In [14]:
result_table = applications.copy()

In [15]:
result_table = result_table[result_table['total_score']>0]

In [16]:
# Clip to valid range 0–100
result_table['total_score'] = result_table['total_score'].clip(lower = 0, upper = 100)

In [17]:
# Verify no scores fall outside the valid range
display((result_table['total_score']<=0).sum())
display((result_table['total_score']>100).sum())

0

0

In [18]:
result_table.set_index('Applied at').resample('W').agg({'total_score' : 'mean'}).round()

,total_score
Applied at,
2022-12-04,51.0
2022-12-11,49.0
2022-12-18,50.0
2022-12-25,47.0
2023-01-01,51.0
2023-01-08,51.0
2023-01-15,52.0
